In [11]:
import pandas as pd 
import numpy as np
import requests
import websocket
import threading

In [76]:
import websocket
import threading
import time

# 全局变量
ws_global = None
response_event = threading.Event()
last_response = None

headers = {"User-Agent":"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/136.0.0.0 Safari/537.36 Edg/136.0.0.0"}
# 转换为 list 形式传入（websocket-client 要求）
header_list = [f"{key}: {value}" for key, value in headers.items()]
# 接收到消息的回调
def on_message(ws, message):
    global last_response
    last_response = message
    response_event.set()  # 通知主线程：消息到了
    print("👂 收到消息:", message)

def on_open(ws):
    print("✅ WebSocket 连接已建立")

def on_error(ws, error):
    print("❌ 错误:", error)

def on_close(ws, close_status_code, close_msg):
    print("🔌 WebSocket 连接已关闭")

# 建立连接
def connect_ws(url):
    global ws_global
    ws = websocket.WebSocketApp(
        url,
        header=header_list,
        on_open=on_open,
        on_message=on_message,
        on_error=on_error,
        on_close=on_close
    )

    ws_global = ws

    thread = threading.Thread(target=ws.run_forever, daemon=True)
    thread.start()

# 支持拿返回值的发送函数
def send_and_wait(msg, timeout=5):
    global last_response
    if not ws_global:
        raise RuntimeError("未连接 WebSocket")

    last_response = None
    response_event.clear()
    ws_global.send(msg)
    print(f"📤 已发送: {msg}")

    # 等待返回值或超时
    received = response_event.wait(timeout)
    if received:
        return last_response
    else:
        return None  # 超时无响应

# 可选的断开函数
def close():
    if ws_global:
        ws_global.close()


In [86]:
connect_ws("ws://192.168.0.166:8899/default-ws")

In [95]:
close()

🔌 WebSocket 连接已关闭


In [87]:
response = send_and_wait("M")
if response:
    print("✔️ 返回值是：", response)
else:
    print("❌ 超时未收到服务器响应")

📤 已发送: M
👂 收到消息: A1489d5df-9f63-493b-bbbf-fbb756b07185
✔️ 返回值是： A1489d5df-9f63-493b-bbbf-fbb756b07185


In [88]:
response = send_and_wait("$479.90000009536743;default;;_OnNamespaceConnect;0;0;")
if response:
    print("✔️ 返回值是：", response)


📤 已发送: $479.90000009536743;default;;_OnNamespaceConnect;0;0;
👂 收到消息: $479.90000009536743;;;;;;
✔️ 返回值是： $479.90000009536743;;;;;;


In [89]:
import json

# 构造后半部分的 JSON 字符串
payload = {
    "initDataRaw": "user=%7B%22id%22%3A7503197642%2C%22first_name%22%3A%22GG%22%2C%22last_name%22%3A%22Bond%22%2C%22language_code%22%3A%22zh-hans%22%2C%22allows_write_to_pm%22%3Atrue%2C%22photo_url%22%3A%22https%3A%5C%2F%5C%2Ft.me%5C%2Fi%5C%2Fuserpic%5C%2F320%5C%2F2x_vITFMhtR509WiaooZ_CP4HFG7VeRj77Axd-CAuRLHLMb3j0pbB6bvJdbx-IAo.svg%22%7D&chat_instance=-4307385486232935476&chat_type=sender&start_param=3-755844&auth_date=1747209703&signature=XeWbDOiG2wHNB4QjQX87YbCUwmOYgEeOtdJ7_5TcH5bk2EgIOkkUsypLa21ofjuiHgo7NxTfikXj_yu-W0-CCw&hash=73812c7242a407b4674bf03f117908fc6f174eae86b51a0f7a57463c9191edf5","provider": "TELEGRAM"
}

# 拼接完整 WebSocket 请求字符串（包含前缀）
full_text =";default;;Auth;0;0;"+  json.dumps(payload)

In [90]:
response = send_and_wait(full_text)

📤 已发送: ;default;;Auth;0;0;{"initDataRaw": "user=%7B%22id%22%3A7503197642%2C%22first_name%22%3A%22GG%22%2C%22last_name%22%3A%22Bond%22%2C%22language_code%22%3A%22zh-hans%22%2C%22allows_write_to_pm%22%3Atrue%2C%22photo_url%22%3A%22https%3A%5C%2F%5C%2Ft.me%5C%2Fi%5C%2Fuserpic%5C%2F320%5C%2F2x_vITFMhtR509WiaooZ_CP4HFG7VeRj77Axd-CAuRLHLMb3j0pbB6bvJdbx-IAo.svg%22%7D&chat_instance=-4307385486232935476&chat_type=sender&start_param=3-755844&auth_date=1747209703&signature=XeWbDOiG2wHNB4QjQX87YbCUwmOYgEeOtdJ7_5TcH5bk2EgIOkkUsypLa21ofjuiHgo7NxTfikXj_yu-W0-CCw&hash=73812c7242a407b4674bf03f117908fc6f174eae86b51a0f7a57463c9191edf5", "provider": "TELEGRAM"}
👂 收到消息: ;default;;Auth;0;0;{
  "rCode": 200,
  "timestamp": 1747218722690,
  "model": {
    "firstOpen": false,
    "player": {
      "inventory": {
        "eggStatistics": 0,
        "freePerch": 0,
        "perch": 0,
        "turtle": 1
      },
      "profile": {
        "id": "0a2222f7-d992-4f5c-a43a-ee26c55e549d",
        "serialNumber": 100

In [94]:
payload={
    "turtles":[],"page":1,"pageSize":10
}

my_turtles= send_and_wait(";default;;Turtles;0;0;"+  json.dumps(payload))

📤 已发送: ;default;;Turtles;0;0;{"turtles": [], "page": 1, "pageSize": 10}
👂 收到消息: ;default;;Turtles;0;0;{
  "rCode": 200,
  "timestamp": 1747218896945,
  "model": {
    "count": 1,
    "page": 1,
    "pageSize": 10,
    "list": [
      {
        "turtle": {
          "id": "31243f2d-47f6-4f1b-a9f7-ebd6103b0d51",
          "serialNumber": 10149,
          "level": 1,
          "genotype": "EEEEEEEE",
          "hashCode": "8E",
          "isFree": true,
          "isJackpot": false,
          "isPromo": false,
          "haveBenefits": false,
          "ownerId": "0a2222f7-d992-4f5c-a43a-ee26c55e549d",
          "hatchedById": "",
          "hatchedAt": 0,
          "hatchedFromInvoiceId": 0,
          "putForSaleAt": 0,
          "saleUpdatedAt": 0,
          "breedFromInvoiceId": 0,
          "farmRewardsClaimed": 0,
          "farmRewardsToClaim": 0,
          "farmRewardsClaimedAt": 0,
          "createdAt": 1747216557000
        },
        "rarity": 0.20412414523193154,
        "eggF